# GPU Training with xaytune

This notebook demonstrates two ways to run GPU-accelerated training with xaytune on OpenShift AI:

1. **Single-GPU Training** — Run LoRA fine-tuning directly in your workbench notebook
2. **Ray Distributed Training** — Scale to multiple GPUs using CodeFlare SDK + KubeRay

Both modes use the exact same `xaytune.finetune()` API. Ray is purely an orchestration layer — no library changes needed.

### Prerequisites

- OpenShift AI workbench with at least 1 NVIDIA GPU
- For Ray section: CodeFlare SDK and KubeRay operator installed on the cluster
- `pip install xaytune` (and `ray[train]` for distributed)

In [ ]:
!pip install xaytune --quiet

In [ ]:
import os
import json
import tempfile
from pathlib import Path

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
print(f"GPUs available:  {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_mem / 1e9
        print(f"  GPU {i}: {name} ({mem:.1f} GB)")

In [ ]:
# Prepare training data — small alpaca-format dataset
TRAIN_DATA = [
    {"instruction": "Translate to French", "input": "Hello, how are you?", "output": "Bonjour, comment allez-vous?"},
    {"instruction": "Summarize the following text", "input": "Machine learning is a subset of artificial intelligence that enables systems to learn from data.", "output": "ML is an AI subset that learns from data."},
    {"instruction": "Write a Python function", "input": "Calculate factorial of n", "output": "def factorial(n):\n    return 1 if n <= 1 else n * factorial(n - 1)"},
    {"instruction": "Explain the concept", "input": "What is gradient descent?", "output": "Gradient descent is an optimization algorithm that iteratively adjusts parameters in the direction of steepest descent of the loss function."},
    {"instruction": "Fix the bug", "input": "for i in range(10):\n    print(i + 1", "output": "for i in range(10):\n    print(i + 1)"},
    {"instruction": "Convert to SQL", "input": "Get all users older than 30", "output": "SELECT * FROM users WHERE age > 30;"},
    {"instruction": "Translate to Spanish", "input": "The weather is nice today", "output": "El clima est\u00e1 agradable hoy"},
    {"instruction": "Write a haiku", "input": "about programming", "output": "Code flows like water\nBugs hide in the deepest depths\nTests reveal the truth"},
    {"instruction": "Simplify this expression", "input": "2x + 3x - x + 4", "output": "4x + 4"},
    {"instruction": "What is the output?", "input": "print(list(range(0, 10, 3)))", "output": "[0, 3, 6, 9]"},
]

DATA_PATH = Path(tempfile.mkdtemp()) / "train.jsonl"
with open(DATA_PATH, "w") as f:
    for sample in TRAIN_DATA:
        f.write(json.dumps(sample) + "\n")

print(f"Training data: {len(TRAIN_DATA)} samples written to {DATA_PATH}")

---

## 1. Single-GPU Training

Run LoRA fine-tuning directly on the workbench GPU. This uses a single GPU with LoRA adapters, keeping VRAM usage around 16 GB for a 7B model.

In [ ]:
import xaytune

# Set HF_TOKEN if using a gated model (e.g. Llama)
# os.environ["HF_TOKEN"] = "hf_..."

MODEL = "mistralai/Mistral-7B-v0.3"

state = xaytune.finetune(
    model=MODEL,
    dataset=str(DATA_PATH),
    method="lora",
    format="alpaca",
    num_epochs=1,
    learning_rate=2e-4,
    batch_size=2,
    max_steps=20,
)

print(f"Loss:  {state.metrics['loss']:.4f}")
print(f"Steps: {state.global_step}")

---

## 2. Ray Distributed Training with CodeFlare SDK

Scale to multiple GPUs by submitting the same training as a Ray job. The workflow:

1. Write a training script that uses Ray's `TorchTrainer` with `xaytune.finetune()` inside
2. Create a `RayCluster` via CodeFlare SDK
3. Submit the script as a `RayJob` via `RayJobClient`
4. Monitor logs and tear down the cluster

**Why this works without changes to xaytune:** xaytune's distributed init reads `RANK`, `WORLD_SIZE`, `LOCAL_RANK` environment variables and skips `torch.distributed.init_process_group()` if it's already initialized. Ray Train sets these same env vars and initializes the process group before calling your training function — so xaytune's setup becomes a no-op inside Ray workers.

In [ ]:
%%writefile train_xaytune.py
"""xaytune distributed fine-tuning via Ray TorchTrainer.

Submit this script to a RayCluster using CodeFlare SDK's RayJobClient.
"""
import json
import tempfile
from pathlib import Path

from ray.train import ScalingConfig
from ray.train.torch import TorchTrainer

TRAIN_DATA = [
    {"instruction": "Translate to French", "input": "Hello, how are you?", "output": "Bonjour, comment allez-vous?"},
    {"instruction": "Summarize the following text", "input": "Machine learning is a subset of artificial intelligence that enables systems to learn from data.", "output": "ML is an AI subset that learns from data."},
    {"instruction": "Write a Python function", "input": "Calculate factorial of n", "output": "def factorial(n):\n    return 1 if n <= 1 else n * factorial(n - 1)"},
    {"instruction": "Explain the concept", "input": "What is gradient descent?", "output": "Gradient descent is an optimization algorithm that iteratively adjusts parameters in the direction of steepest descent of the loss function."},
    {"instruction": "Fix the bug", "input": "for i in range(10):\n    print(i + 1", "output": "for i in range(10):\n    print(i + 1)"},
    {"instruction": "Convert to SQL", "input": "Get all users older than 30", "output": "SELECT * FROM users WHERE age > 30;"},
    {"instruction": "Translate to Spanish", "input": "The weather is nice today", "output": "El clima est\u00e1 agradable hoy"},
    {"instruction": "Write a haiku", "input": "about programming", "output": "Code flows like water\nBugs hide in the deepest depths\nTests reveal the truth"},
    {"instruction": "Simplify this expression", "input": "2x + 3x - x + 4", "output": "4x + 4"},
    {"instruction": "What is the output?", "input": "print(list(range(0, 10, 3)))", "output": "[0, 3, 6, 9]"},
]

MODEL = "mistralai/Mistral-7B-v0.3"


def train_fn():
    import xaytune
    from ray.train import report

    data_path = Path(tempfile.mkdtemp()) / "train.jsonl"
    with open(data_path, "w") as f:
        for s in TRAIN_DATA:
            f.write(json.dumps(s) + "\n")

    state = xaytune.finetune(
        model=MODEL,
        dataset=str(data_path),
        method="lora",
        format="alpaca",
        num_epochs=1,
        learning_rate=2e-4,
        batch_size=2,
        max_steps=20,
    )
    report({"loss": state.metrics.get("loss", 0), "steps": state.global_step})


if __name__ == "__main__":
    trainer = TorchTrainer(
        train_fn,
        scaling_config=ScalingConfig(
            num_workers=2,
            use_gpu=True,
            resources_per_worker={"GPU": 1},
        ),
    )
    result = trainer.fit()
    print(f"Training complete. Metrics: {result.metrics}")

In [ ]:
from codeflare_sdk import Cluster, ClusterConfiguration

NUM_WORKERS = 2  # adjust to your available GPUs

cluster = Cluster(ClusterConfiguration(
    name="xaytune-training",
    num_workers=NUM_WORKERS,
    head_cpu_requests=4,
    head_cpu_limits=4,
    head_memory_requests=16,
    head_memory_limits=16,
    worker_cpu_requests=4,
    worker_cpu_limits=4,
    worker_memory_requests=32,
    worker_memory_limits=32,
    head_extended_resource_requests={"nvidia.com/gpu": 0},
    worker_extended_resource_requests={"nvidia.com/gpu": 1},
    # image="quay.io/modh/ray:2.35.0-py311-cu121",
    write_to_file=False,
    # local_queue="your-local-queue",
))

In [ ]:
cluster.apply()
cluster.wait_ready()
cluster.details()

In [ ]:
import time
from codeflare_sdk import RayJobClient

client = RayJobClient(
    address=cluster.cluster_dashboard_uri(),
    verify=False,
)

submission_id = client.submit_job(
    entrypoint="python train_xaytune.py",
    runtime_env={
        "working_dir": "./",
        "pip": ["xaytune", "transformers", "peft", "bitsandbytes", "datasets"],
        "env_vars": {
            # "HF_TOKEN": "hf_...",  # uncomment for gated models
        },
    },
)
print(f"Submitted job: {submission_id}")

while True:
    status = client.get_job_status(submission_id)
    print(f"Status: {status}")
    if status in {"SUCCEEDED", "FAILED", "STOPPED"}:
        break
    time.sleep(15)

print("\n--- Job Logs ---")
print(client.get_job_logs(submission_id))

In [ ]:
cluster.down()
print("Ray cluster torn down.")

---

## 3. Configuration Variants

Swap these into the training cells above depending on your hardware and model.

In [ ]:
# --- QLoRA: Fine-tune larger models on smaller GPUs ---
# Uses 4-bit quantization + LoRA, fits 70B models on 24 GB GPUs
state = xaytune.finetune(
    model="mistralai/Mistral-7B-v0.3",
    dataset=str(DATA_PATH),
    method="qlora",
    format="alpaca",
    num_epochs=1,
    learning_rate=2e-4,
    batch_size=2,
    max_steps=20,
)

In [ ]:
# --- Full config with FSDP strategy (for multi-GPU without Ray) ---
# Launch with: torchrun --nproc_per_node=2 your_script.py
from xaytune.config.schema import (
    DataConfig,
    FSDPConfig,
    LoraConfig,
    ModelConfig,
    OutputConfig,
    TrainConfig,
    TrainerConfig,
)

config = TrainConfig(
    recipe="finetune",
    method="lora",
    model=ModelConfig(name="mistralai/Mistral-7B-v0.3", dtype="bfloat16"),
    data=DataConfig(path=str(DATA_PATH), format="alpaca", max_seq_length=512),
    lora=LoraConfig(rank=16, alpha=32),
    trainer=TrainerConfig(
        strategy="fsdp",
        batch_size=2,
        learning_rate=2e-4,
        num_epochs=1,
        max_steps=20,
        mixed_precision="bf16",
    ),
    fsdp=FSDPConfig(
        sharding_strategy="full_shard",
        activation_checkpointing=True,
    ),
    output=OutputConfig(dir="/tmp/fsdp-output"),
)

state = xaytune.finetune(config=config)
print(f"Loss: {state.metrics['loss']:.4f}")

---

## Notes

### Choosing `num_workers`
Set `num_workers` in `ScalingConfig` and `ClusterConfiguration` to match your available GPUs. Each worker gets 1 GPU.

### FSDP vs DDP
- **DDP** (Data Distributed Parallel): Each GPU holds a full model copy. Best for models that fit in a single GPU's VRAM.
- **FSDP** (Fully Sharded Data Parallel): Shards model parameters across GPUs. Use for models > 7B or when VRAM is tight.
- xaytune defaults to `strategy="auto"` which picks FSDP for multi-GPU and no wrapping for single-GPU.

### Worker memory sizing
- LoRA 7B model: ~16 GB GPU VRAM, ~32 GB RAM per worker
- QLoRA 7B model: ~8 GB GPU VRAM, ~16 GB RAM per worker
- Full fine-tune 7B: ~28 GB GPU VRAM per worker (use FSDP to shard)

### Gated models
For models like Llama that require access approval, set `HF_TOKEN`:
```python
os.environ["HF_TOKEN"] = "hf_..."
```
For Ray jobs, pass it via `runtime_env["env_vars"]` in `submit_job()`.

### GPU type
If using AMD GPUs instead of NVIDIA, replace `nvidia.com/gpu` with `amd.com/gpu` in `ClusterConfiguration`.